In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType, TimestampType
from pyspark.sql import Row

catalog = "Ecommerce_BI"

In [0]:
df_bronze = spark.read.table(f'{catalog}.bronze.brz_order_items')

df_bronze.show(5)

##Cleaning

In [0]:
df_bronze.describe().show()
df_bronze.printSchema()

In [0]:
# Removing duplicates on primary keys
df_bronze = df_bronze.dropDuplicates(["order_id", "item_seq"])

# Cleaning the quantity column
df_bronze = df_bronze.withColumn("quantity",
    F.when(F.col("quantity") == "Two", 2).otherwise(F.col("quantity").cast("int"))
)

# Cleaning unit price
df_bronze = df_bronze.withColumn("unit_price",
    F.regexp_replace(F.col("unit_price"), "[$]", "").cast("float")
)

# Cleaning discount percentage
df_bronze = df_bronze.withColumn("discount_pct",
    F.regexp_replace(F.col("discount_pct"), "%", "").cast("double")
)
# Enhancing channel processing
df_bronze = df_bronze.withColumn("channel",
    F.when(F.col("channel") == "app", "Mobile")
    .when(F.col("channel") == "web", "Website")
    .otherwise(F.col("channel"))
)

df_bronze.show(5)

In [0]:
# Transformation: datatype conversions
# Convert dt (string → date)
df_bronze = df_bronze.withColumn(
    "dt",
    F.to_date("dt", "yyyy-MM-dd")     
)

# Convert order_ts (string → timestamp)
df_bronze = df_bronze.withColumn(
    "order_ts",
    F.coalesce(
        F.to_timestamp("order_ts", "yyyy-MM-dd HH:mm:ss"),  # matches 2025-08-01 22:53:52
        F.to_timestamp("order_ts", "dd-MM-yyyy HH:mm")      # fallback for 01-08-2025 22:53
    )
)


# Convert item_seq (string → integer)
df_bronze = df_bronze.withColumn(
    "item_seq",
    F.col("item_seq").cast("int")
)

# Convert tax_amount (string → double, strip non-numeric characters)
df_bronze = df_bronze.withColumn(
    "tax_amount",
    F.regexp_replace("tax_amount", r"[^0-9.\-]", "").cast("double")
)


# Add processed time 
df_silver = df_bronze.withColumn(
    "processed_time", F.current_timestamp()
)

In [0]:
df_silver.printSchema()

In [0]:
df_silver.write.format('delta').mode('overwrite').option('mergeSchema', True).saveAsTable(f'{catalog}.silver.slvr_order_items')